# L4 Lecture Notes: Linear Regression, OLS, and Ridge Regression

This notebook is a self-contained replacement for the L4 slides. It develops every lecture topic, fills in the animated proof steps, solves the slide exercises, and adds the most relevant results from the two accompanying books.

**Primary sources**

- *L4: Linear regression, OLS, ridge regression*, slides 3-37 (substantive physical PDF pages 4-176; the PDF contains many repeated animation frames).
- Francis Bach, *Learning Theory from First Principles*, Chapter 3, book pages 45-66 (physical PDF pages 61-82), with selected extensions from Sections 12.1-12.2 and 14.1.
- Shalev-Shwartz and Ben-David, *Understanding Machine Learning: Solution Manual*, physical/book pages 26 and 64-65: absolute-loss regression as a linear program, the rank criterion, centering with an intercept, and scalar soft-thresholding.

**How to read the notebook**

Sections 1-7 build the regression problem and the one-dimensional example. Sections 8-15 derive and analyze OLS. Sections 16-22 develop ridge and general regularization. Sections 23-24 are advanced extensions that clarify high-dimensional and random-design behavior. Sections 25-26 collect worked exercises and a final checklist.

**Formula convention.** Inline mathematics uses `$...$`; displayed mathematics uses `$$...$$`. Throughout, $N$ is the sample size and $d$ is the number of fitted parameters, including an intercept if it is represented as a feature. The books often use $n$ where this notebook uses $N$.


## 1. Regression setup and notation

A supervised sample is

$$
S=\{(x_1,y_1),\ldots,(x_N,y_N)\},
\qquad x_i\in\mathbb{R}^p,\quad y_i\in\mathbb{R}.
$$

Classification predicts a category such as $y\in\{0,1\}$; regression predicts a numerical response. We seek a rule $h:\mathbb{R}^p\to\mathbb{R}$ that predicts the response of a new input.

A feature map $\phi:\mathbb{R}^p\to\mathbb{R}^d$ produces the feature vector used by the model. A predictor that is linear in its parameters has the form

$$
f_w(x)=\phi(x)^\top w,
\qquad w\in\mathbb{R}^d.
$$

Put the feature vectors into the design matrix and responses into a vector:

$$
X=
\begin{bmatrix}
\phi(x_1)^\top\\
\vdots\\
\phi(x_N)^\top
\end{bmatrix}\in\mathbb{R}^{N\times d},
\qquad
y=
\begin{bmatrix}y_1\\ \vdots\\ y_N\end{bmatrix}\in\mathbb{R}^N.
$$

Then the vector of fitted responses is $Xw$. An affine model $w_0+x^\top\beta$ is included by taking $\phi(x)=(1,x^\top)^\top$ and $w=(w_0,\beta^\top)^\top$. Unless stated otherwise, an intercept is assumed to be included this way.

The noncentered empirical feature covariance is

$$
\widehat\Sigma=\frac{1}{N}X^\top X.
$$

Do not confuse the input dimension $p$, the feature dimension $d$, and the sample size $N$.

*Sources: L4 slides 3-4; Bach, Sections 3.1-3.2.*


## 2. Empirical risk minimization and loss choices

For a loss $\ell(\widehat y,y)$ and hypothesis class $\mathcal H$, empirical risk minimization (ERM) solves

$$
\widehat h\in\arg\min_{h\in\mathcal H}
\widehat R(h),
\qquad
\widehat R(h)=\frac1N\sum_{i=1}^N\ell(h(x_i),y_i).
$$

**Squared loss** gives least-squares regression:

$$
\ell(\widehat y,y)=(\widehat y-y)^2,
\qquad
\widehat w\in\arg\min_w\frac1N\|Xw-y\|_2^2.
$$

It is convex and differentiable, heavily penalizes large residuals, has a closed-form solution for a linear-in-parameters model, equals maximum likelihood under Gaussian noise, and targets the conditional mean at the population level.

**Absolute loss** is more robust to large residuals and targets a conditional median:

$$
\min_w\frac1N\sum_{i=1}^N|y_i-\phi(x_i)^\top w|.
$$

It can be written as a linear program by introducing $s_i\ge 0$:

$$
\begin{aligned}
\min_{w,s}\quad &\sum_{i=1}^N s_i\\
\text{subject to}\quad
&\phi(x_i)^\top w-y_i\le s_i,\\
&y_i-\phi(x_i)^\top w\le s_i,
\qquad i=1,\ldots,N.
\end{aligned}
$$

**Check (pinball) loss** for $\tau\in(0,1)$ is

$$
\rho_\tau(u)=u\bigl(\tau-\mathbf 1\{u<0\}\bigr).
$$

Minimizing $\sum_i\rho_\tau(y_i-h(x_i))$ performs quantile regression and targets the conditional $\tau$-quantile. The slide calls this a leaky-ReLU-shaped loss; in statistics, check or pinball loss is the standard name. Different losses answer different statistical questions, so squared loss is a modeling choice, not a universal default.

*Sources: L4 slide 4; Bach, Sections 2.2.3 and 3.2; solution manual, Chapter 9, Solution 1 (p. 26).*


## 3. Simple linear regression: complete derivation

For a scalar input, consider

$$
\widehat y_i=w_0+wx_i,
\qquad
L(w_0,w)=\sum_{i=1}^N(y_i-w_0-wx_i)^2.
$$

The first-order conditions are

$$
\frac{\partial L}{\partial w_0}
=-2\sum_{i=1}^N(y_i-w_0-wx_i)=0,
$$

$$
\frac{\partial L}{\partial w}
=-2\sum_{i=1}^N x_i(y_i-w_0-wx_i)=0.
$$

Define $\bar x=N^{-1}\sum_i x_i$ and $\bar y=N^{-1}\sum_i y_i$. The first equation gives

$$
\widehat w_0=\bar y-\widehat w\bar x.
$$

After substitution into the second equation,

$$
\widehat w
=\frac{\sum_{i=1}^N(x_i-\bar x)(y_i-\bar y)}
{\sum_{i=1}^N(x_i-\bar x)^2}
=\frac{\widehat{\operatorname{Cov}}(X,Y)}
{\widehat{\operatorname{Var}}(X)}.
$$

If $s_x$ and $s_y$ use the same denominator as the empirical covariance and $r$ is the Pearson correlation, then

$$
\widehat w=r\frac{s_y}{s_x},
\qquad
\widehat y-\bar y=r\frac{s_y}{s_x}(x-\bar x).
$$

Consequences:

- the fitted line passes through $(\bar x,\bar y)$;
- centering both variables removes the intercept without changing the slope;
- changing units changes the slope but not the correlation;
- the slope is undefined if every $x_i$ is identical.

The centering identity is exact:

$$
\min_{w_0,w}\sum_i(y_i-w_0-wx_i)^2
=\min_w\sum_i\bigl[(y_i-\bar y)-w(x_i-\bar x)\bigr]^2.
$$

*Sources: L4 slides 6-7; solution manual, Chapter 25, Solution 1 (p. 64).*


## 4. Pearson's father-son example, regression to the mean, and RMSE

The lecture uses approximate values

$$
\bar x=68,\quad s_x=2.7,
\qquad
\bar y=69,\quad s_y=2.7,
\qquad r=0.5,
$$

where $x$ is a father's height and $y$ is his son's height, in inches. Hence

$$
\widehat w=r\frac{s_y}{s_x}=0.5,
\qquad
\widehat w_0=\bar y-\widehat w\bar x=35,
$$

so $\widehat y=35+0.5x$. For a father of height $64$ inches,

$$
\widehat y=35+0.5(64)=67.
$$

The prediction is not $65$. A father who is $(64-68)/2.7$ standard deviations below the mean predicts a son only $r$ times as many standard deviations below his mean. When $|r|<1$, the predicted standardized response is closer to zero; this is regression toward the mean, not a statement that individuals physically change toward the mean.

For the identities below, use the $1/N$ convention

$$
s_x^2=\frac1N\sum_{i=1}^N(x_i-\bar x)^2,\qquad
s_y^2=\frac1N\sum_{i=1}^N(y_i-\bar y)^2,\qquad
r=\frac{N^{-1}\sum_{i=1}^N(x_i-\bar x)(y_i-\bar y)}{s_xs_y}.
$$

For least squares with an intercept, the empirical residual mean square then satisfies

$$
\frac1N\sum_{i=1}^N(y_i-\widehat w_0-\widehat wx_i)^2
=s_y^2(1-r^2).
$$

Therefore

$$
\operatorname{RMSE}=s_y\sqrt{1-r^2}.
$$

To verify it, center $x$ and $y$ and substitute $\widehat w=r s_y/s_x$:

$$
\begin{aligned}
\operatorname{MSE}
&=\frac1N\sum_i\bigl[(y_i-\bar y)-\widehat w(x_i-\bar x)\bigr]^2\\
&=s_y^2-2\widehat w\,r s_xs_y+\widehat w^2s_x^2\\
&=s_y^2(1-r^2).
\end{aligned}
$$

If $s_y^2$ is instead the conventional sample variance with denominator $N-1$, the same residual MSE is $\frac{N-1}{N}s_y^2(1-r^2)$. This identity is an in-sample identity for simple OLS. A test RMSE on new data need not equal this value.

*Sources: L4 slide 8.*


## 5. Why correlation and summary statistics are not enough

Pearson correlation measures linear association, not causation. A large $|r|$ can arise from confounding, common time trends, selection effects, a restricted measurement range, or a single influential point. Conversely, a strong nonlinear relation can have $r\approx0$.

Anscombe's quartet contains datasets with essentially the same sample means, variances, correlation, and OLS line but visibly different geometry: one is roughly linear, one is curved, one is dominated by an outlier, and one contains a high-leverage point. The slide also shows that arbitrary point clouds can be constructed with matching summaries.

A regression analysis should therefore inspect at least:

- a scatter plot of response against each important feature;
- residuals $e_i=y_i-\widehat y_i$ versus fitted values;
- extreme residuals and high-leverage inputs;
- changing residual spread, which suggests heteroscedasticity;
- nonlinear patterns and subgroup structure;
- whether extrapolation is being attempted outside the observed feature range.

OLS summarizes a conditional association. It does not, by itself, identify an intervention effect or justify a causal conclusion.

*Sources: L4 slides 9-10.*


## 6. Population regression: why the conditional mean is optimal

The population squared-loss risk of a measurable predictor $h$ is

$$
R(h)=\mathbb E\bigl[(Y-h(X))^2\bigr].
$$

Let

$$
\eta(x)=\mathbb E[Y\mid X=x].
$$

For any fixed $x$ and any value $z$,

$$
\begin{aligned}
\mathbb E[(Y-z)^2\mid X=x]
&=\mathbb E[(Y-\eta(x)+\eta(x)-z)^2\mid X=x]\\
&=\operatorname{Var}(Y\mid X=x)+(z-\eta(x))^2,
\end{aligned}
$$

because $\mathbb E[Y-\eta(x)\mid X=x]=0$. The first term does not depend on $z$, and the second is nonnegative. Thus

$$
h^*(x)=\eta(x)=\mathbb E[Y\mid X=x]
$$

minimizes conditional risk for every $x$, and therefore minimizes $R(h)$. More generally,

$$
R(h)=R(h^*)+\mathbb E\bigl[(h(X)-h^*(X))^2\bigr],
$$

with irreducible Bayes risk

$$
R(h^*)=\mathbb E\bigl[\operatorname{Var}(Y\mid X)\bigr].
$$

Geometrically, conditional expectation is the orthogonal projection of $Y$ onto the closed $L^2$ subspace of functions of $X$; the risk identity above is its Pythagorean theorem.

The joint distribution of $(X,Y)$ is unknown, so $\eta(x)$ cannot usually be computed directly. Learning restricts $h$ to a tractable class and approximates population risk by empirical risk. If $\eta$ is outside the class, the remaining gap is approximation error or model misspecification.

*Sources: L4 slides 11-13; Bach, Sections 2.2.3 and 3.2.*


## 7. Discriminative regression, nonlinear features, and misspecification

A discriminative regression model describes $Y$ given $X$ directly, rather than modeling the full joint distribution $\mathbb P(X,Y)$. A linear feature model assumes

$$
\mathbb E[Y\mid X=x]\approx f_w(x)=\phi(x)^\top w.
$$

The word **linear** refers to $w$, not necessarily to $x$. Examples include

$$
\phi(x)=(1,x,x^2,\ldots,x^k)^\top
$$

for polynomial regression, splines, interactions, periodic features, or a fixed representation learned elsewhere. The resulting curve may be highly nonlinear in $x$ while OLS remains a quadratic optimization problem in $w$.

The learning problem separates three choices:

1. the feature map $\phi$, which controls the functions available;
2. the loss, which determines the population target;
3. the algorithm or regularizer, which selects among parameters and controls stability.

A larger feature space reduces approximation error but may increase estimation variance and overfitting. Training error can only decrease when nested feature classes are enlarged, but test error need not. This is the first appearance of the bias-variance trade-off.

The binary VC dimension used for $0/1$ classification does not directly analyze unbounded squared loss. Real-valued capacity tools include pseudo-dimension and Rademacher complexity. The slides mention this issue but defer the general theory; this lecture instead obtains an explicit finite-sample calculation under a linear fixed-design model.

*Sources: L4 slides 15-17; Bach, Sections 2.3.2 and 3.2.*


## 8. Multivariate OLS and the normal equations

Ordinary least squares (OLS) minimizes

$$
\widehat R(w)=\frac1N\|y-Xw\|_2^2.
$$

Expanding gives

$$
\widehat R(w)=\frac1N
\left(y^\top y-2w^\top X^\top y+w^\top X^\top Xw\right).
$$

Hence

$$
\nabla\widehat R(w)=\frac2N(X^\top Xw-X^\top y),
\qquad
\nabla^2\widehat R(w)=\frac2N X^\top X=2\widehat\Sigma.
$$

Setting the gradient to zero yields the normal equations

$$
X^\top X\widehat w=X^\top y.
$$

If $X$ has full column rank $d$ (so necessarily $d\le N$), then $X^\top X$ is positive definite and the minimizer is unique:

$$
\widehat w_{\mathrm{OLS}}=(X^\top X)^{-1}X^\top y.
$$

The rank criterion is

$$
X^\top X\text{ is invertible}
\quad\Longleftrightarrow\quad
\operatorname{rank}(X)=d
\quad\Longleftrightarrow\quad
\text{the columns of }X\text{ are linearly independent}.
$$

Indeed, $u^\top X^\top Xu=\|Xu\|_2^2$, so $\operatorname{null}(X^\top X)=\operatorname{null}(X)$.

Perfect multicollinearity violates this condition. Near-collinearity does not destroy uniqueness but makes the estimate numerically and statistically unstable.

*Sources: L4 slide 16; Bach, Sections 3.2-3.3.1; solution manual, Chapter 9, Solution 2 (p. 26).*


## 9. Projection geometry, the hat matrix, and stable computation

Define

$$
H=X(X^\top X)^{-1}X^\top.
$$

For full column rank, $H^\top=H$ and $H^2=H$, so $H$ is the orthogonal projector onto the column space $\operatorname{col}(X)$. The fitted response and residual are

$$
\widehat y=Hy,
\qquad
e=y-\widehat y=(I-H)y.
$$

The normal equations are equivalent to

$$
X^\top e=0,
$$

so the residual is orthogonal to every fitted feature direction. The diagonal $h_{ii}$ is the leverage of observation $i$; unusually large leverage means that its feature vector can strongly influence its own fit.

The inverse formula is useful for algebra, but numerical code should not explicitly form $(X^\top X)^{-1}$. Forming $X^\top X$ squares the spectral condition number:

$$
\kappa_2(X^\top X)=\kappa_2(X)^2.
$$

Preferred methods are:

- QR factorization $X=QR$, followed by $R\widehat w=Q^\top y$;
- SVD or a least-squares routine, especially near rank deficiency;
- iterative least-squares methods such as LSQR or LSMR for very large problems; they need only products with $X$ and $X^\top$. Conjugate gradient is appropriate for a suitable symmetric positive-definite system, such as a ridge system, usually with preconditioning; applying it directly to $X^\top X$ inherits the squared condition number.

In Python/NumPy, the conceptual recommendation is `numpy.linalg.lstsq(X, y)`, not `inv(X.T @ X) @ X.T @ y`.

*Sources: Bach, Sections 3.3.2-3.3.3; projection details expand the animated OLS slide.*


## 10. Fixed design, random design, and the observation model

To analyze an estimator, specify what is random.

**Fixed design:** the rows of $X$ are treated as deterministic. Only the responses vary. Performance is evaluated at the same input points, possibly with freshly sampled responses. This is an in-sample denoising problem.

**Random design:** training inputs are random, and performance is evaluated on a new input drawn from the population. This is the usual generalization problem.

The lecture begins with the fixed-design, well-specified linear model

$$
y=Xw_*+\varepsilon,
\qquad
\mathbb E[\varepsilon]=0,
\qquad
\operatorname{Cov}(\varepsilon)=\sigma^2I_N.
$$

The assumptions mean:

- **well-specified:** the conditional mean at the design points equals $Xw_*$;
- **zero mean:** errors do not systematically shift the response;
- **homoscedastic:** every response has variance $\sigma^2$;
- **uncorrelated:** off-diagonal noise covariances vanish.

Independence is an additional assumption: $\operatorname{Cov}(\varepsilon)=\sigma^2I_N$ does not imply it. Independent Gaussian errors with the stated covariance are iid, but the OLS mean, covariance, and expected fixed-design risk require only the displayed first and second moments (and a fresh test-noise vector independent of the training noise).

Gaussianity is stronger than needed for those OLS moment and risk calculations. It is needed for exact likelihood statements and some exact distributional results. Under model misspecification, an approximation-error term remains; under heteroscedastic or correlated noise with covariance $C$, the covariance formulas use $C$ instead of $\sigma^2I$.

*Sources: L4 slide 18; Bach, Sections 3.4-3.5.*


## 11. Gaussian likelihood and two estimators of noise variance

If the fixed-design noise is Gaussian,

$$
\varepsilon_i\overset{\mathrm{iid}}\sim\mathcal N(0,\sigma^2),
$$

then

$$
p(y\mid w,\sigma^2)
=(2\pi\sigma^2)^{-N/2}
\exp\left(-\frac{\|y-Xw\|_2^2}{2\sigma^2}\right).
$$

For fixed $\sigma^2$, maximizing likelihood is equivalent to minimizing residual sum of squares, so the maximum-likelihood estimate of $w$ is OLS. Provided the minimized residual sum of squares is positive, joint maximization over $w$ and $\sigma^2$ gives

$$
\widetilde\sigma^2_{\mathrm{MLE}}
=\frac1N\|y-X\widehat w\|_2^2.
$$

This MLE is biased downward because $d$ fitted parameters have already used some noise degrees of freedom. Under full column rank and $N>d$, the unbiased estimator is

$$
\widehat\sigma^2
=\frac{\|y-X\widehat w\|_2^2}{N-d}.
$$

These formulas answer different questions: division by $N$ comes from maximum likelihood; division by $N-d$ corrects the expectation. With full column rank and $N>d$, Gaussian noise makes the minimized RSS positive almost surely. If $N=d$, OLS fits exactly, RSS is zero, $\mathrm{RSS}/(N-d)$ is undefined, and the likelihood becomes unbounded as $\sigma^2\downarrow0$, so there is no joint MLE in the domain $\sigma^2>0$. Neither formula is valid unchanged when the effective rank differs from $d$ or when the variance model is not homoscedastic.

*Sources: L4 slide 18 (MLE remark); Bach, Section 3.5 and Exercises 3.1-3.2.*


## 12. Fixed-design risk and its bias-variance decomposition

Let a fresh response at the same design points be

$$
y'=Xw_*+\varepsilon',
\qquad
\varepsilon'\text{ independent of the training noise}.
$$

Define fixed-design test risk

$$
R_X(w)=\mathbb E_{\varepsilon'}\left[\frac1N\|y'-Xw\|_2^2\right].
$$

Since the cross term has zero expectation,

$$
R_X(w)=\sigma^2+\frac1N\|X(w-w_*)\|_2^2
=\sigma^2+\|w-w_*\|_{\widehat\Sigma}^2,
$$

where $\|v\|_{\widehat\Sigma}^2=v^\top\widehat\Sigma v$. Thus $R_X^*=\sigma^2$ and

$$
R_X(w)-R_X^*=\|w-w_*\|_{\widehat\Sigma}^2.
$$

For a random estimator $\widetilde w$, add and subtract $\mathbb E[\widetilde w]$:

$$
\begin{aligned}
\mathbb E[R_X(\widetilde w)]-R_X^*
&=\|\mathbb E[\widetilde w]-w_*\|_{\widehat\Sigma}^2\\
&\quad+\mathbb E\|\widetilde w-\mathbb E[\widetilde w]\|_{\widehat\Sigma}^2.
\end{aligned}
$$

The first term is squared bias and the second is variance, both measured in the prediction geometry induced by $X$. This is more relevant than Euclidean parameter error when prediction, rather than parameter recovery, is the goal.

*Source: Bach, Proposition 3.3.*


## 13. OLS unbiasedness, covariance, and exact prediction error

Under the fixed-design model and full column rank,

$$
\widehat w
=(X^\top X)^{-1}X^\top(Xw_*+\varepsilon)
=w_*+(X^\top X)^{-1}X^\top\varepsilon.
$$

Hence

$$
\mathbb E[\widehat w]=w_*,
$$

and

$$
\operatorname{Cov}(\widehat w)
=\sigma^2(X^\top X)^{-1}
=\frac{\sigma^2}{N}\widehat\Sigma^{-1}.
$$

Small eigenvalues of $X^\top X$ therefore create large parameter variance. For predictions, use the projector $H$:

$$
X\widehat w-Xw_*=H\varepsilon.
$$

Since $H^2=H$, $H^\top=H$, and $\operatorname{tr}(H)=d$,

$$
\begin{aligned}
\mathbb E\left[\frac1N\|X\widehat w-Xw_*\|_2^2\right]
&=\frac1N\operatorname{tr}\left(H\,\mathbb E[\varepsilon\varepsilon^\top]\right)\\
&=\frac{\sigma^2d}{N}.
\end{aligned}
$$

Therefore

$$
\mathbb E[R_X(\widehat w)]-R_X^*=\frac{\sigma^2d}{N}.
$$

The animated slide proves the looser bound $2\sigma^2\operatorname{tr}(H)/N$. Under the stated homoscedastic fixed-design assumptions, projector algebra gives the exact result above and removes the factor $2$. The conclusion still requires $d/N\to0$ for the excess risk to vanish.

*Sources: L4 slides 19-25; Bach, Propositions 3.4-3.5.*


## 14. Rank-deficient OLS and the Moore-Penrose pseudoinverse

If $X^\top X$ is singular, OLS predictions still exist, but the parameter minimizer may not be unique. Let the compact SVD be

$$
X=U_r\Sigma_rV_r^\top,
\qquad
\Sigma_r=\operatorname{diag}(s_1,\ldots,s_r),
$$

where $r=\operatorname{rank}(X)$. The Moore-Penrose pseudoinverse is

$$
X^+=V_r\Sigma_r^{-1}U_r^\top.
$$

The minimum-Euclidean-norm least-squares solution is

$$
\widehat w_{\min}=X^+y,
$$

and the fitted vector is unique:

$$
X\widehat w_{\min}=XX^+y=H_ry,
\qquad
H_r=U_rU_r^\top.
$$

$H_r$ is the projector onto $\operatorname{col}(X)$ and $\operatorname{tr}(H_r)=r$. Therefore the exact same-design signal-prediction error is

$$
\mathbb E\left[\frac1N\|X\widehat w-Xw_*\|_2^2\right]
=\frac{\sigma^2r}{N}.
$$

Parameter recovery is different: components of $w_*$ in $\operatorname{null}(X)$ are unidentifiable. Any vector $\widehat w_{\min}+v$ with $v\in\operatorname{null}(X)$ has the same training predictions. Saying “the OLS solution” in a rank-deficient problem is incomplete unless the selection rule is specified.

*Sources: L4 slide 25 exercise; Bach, Sections 3.3 and 12.1.1.*


## 15. Training error, test error, and optimism

The training residual is

$$
y-X\widehat w=(I-H)\varepsilon.
$$

Because $I-H$ is a projector of rank $N-d$,

$$
\mathbb E\left[\frac1N\|y-X\widehat w\|_2^2\right]
=\sigma^2\left(1-\frac dN\right).
$$

By contrast, the expected loss on an independent response $y'$ at the same design is

$$
\mathbb E\left[\frac1N\|y'-X\widehat w\|_2^2\right]
=\sigma^2+\frac{\sigma^2d}{N}.
$$

Thus expected training loss is optimistically low by

$$
\frac{2\sigma^2d}{N}.
$$

This explains three facts at once:

- increasing model dimension decreases training error even when it only fits noise;
- training error is not an unbiased estimate of predictive risk;
- the residual degrees-of-freedom correction gives $\|y-X\widehat w\|^2/(N-d)$ as an unbiased estimator of $\sigma^2$.

These equalities are for full-rank fixed design. For rank $r$, replace $d$ by $r$. For random design or model misspecification, additional terms appear.

*Source: Bach, Section 3.5.1 and Exercise 3.2.*


## 16. Why regularize, and how ridge regression is derived

OLS becomes fragile when $d\ge N$, when features are nearly collinear, or when a flexible feature map fits noise. Ridge regression adds an $\ell_2$ penalty. This notebook uses the single convention

$$
\widehat w_\lambda
=\arg\min_w
\left\{
\frac1N\|y-Xw\|_2^2+\lambda\|w\|_2^2
\right\},
\qquad \lambda>0.
$$

Differentiating and setting the gradient to zero gives

$$
\frac2N X^\top(Xw-y)+2\lambda w=0,
$$

so

$$
\boxed{
\widehat w_\lambda=(X^\top X+N\lambda I)^{-1}X^\top y
=\frac1N(\widehat\Sigma+\lambda I)^{-1}X^\top y.
}
$$

For every nonzero $z$,

$$
z^\top(X^\top X+N\lambda I)z
=\|Xz\|_2^2+N\lambda\|z\|_2^2>0,
$$

so the ridge system has a unique solution even when $X^\top X$ is singular.

**Scaling warning.** If the loss is written without $1/N$, the solution contains $\lambda I$ instead of $N\lambda I$ after the penalty is rescaled. If factors $1/2$ are added consistently, they cancel. The symbol $\lambda$ has no comparable numerical meaning until the full objective convention is known; this reconciles the different conventions visible across the slide frames.

*Sources: L4 slides 26-29 and 33; Bach, Section 3.6.*


## 17. Ridge as spectral shrinkage, a dual method, and effective dimension

Let $X=U_r\Sigma_rV_r^\top$ with singular values $s_j>0$. Then

$$
\widehat w_\lambda
=V_r\operatorname{diag}\left(\frac{s_j}{s_j^2+N\lambda}\right)U_r^\top y.
$$

Ridge predictions are

$$
\widehat y_\lambda=H_\lambda y,
\qquad
H_\lambda=X(X^\top X+N\lambda I)^{-1}X^\top,
$$

with spectral factors

$$
\frac{s_j^2}{s_j^2+N\lambda}
=\frac{\mu_j}{\mu_j+\lambda},
\qquad
\mu_j=\frac{s_j^2}{N}.
$$

Weakly observed directions with small $s_j$ are shrunk most. This stabilizes nearly singular problems.

The identity

$$
(X^\top X+N\lambda I)^{-1}X^\top
=X^\top(XX^\top+N\lambda I)^{-1}
$$

gives the dual form

$$
\widehat w_\lambda=X^\top(XX^\top+N\lambda I)^{-1}y.
$$

Solving an $N\times N$ system can be cheaper than a $d\times d$ system when $d\gg N$.

Two related quantities are called effective degrees of freedom in different texts:

$$
\operatorname{tr}(H_\lambda)
=\sum_j\frac{\mu_j}{\mu_j+\lambda},
$$

and the variance degrees of freedom

$$
\operatorname{tr}(H_\lambda^2)
=\sum_j\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

The second quantity is the soft parameter count appearing in the exact ridge variance in Bach's text.

*Source: Bach, Section 3.6 and Exercise 3.5.*


## 18. The general bias-variance decomposition

Let $f^*(x)=\mathbb E[Y\mid X=x]$. A training set $D$ makes the learned predictor $\widehat f_D(x)$ random. For an independent test response at a fixed $x$,

$$
Y_0=f^*(x)+\varepsilon_0,
\qquad
\mathbb E[\varepsilon_0\mid x]=0,
\qquad
\operatorname{Var}(\varepsilon_0\mid x)=\sigma^2(x).
$$

Add and subtract $\mathbb E_D[\widehat f_D(x)]$. Independence of test noise and training data makes all cross terms vanish:

$$
\boxed{
\mathbb E_{D,Y_0}\bigl[(Y_0-\widehat f_D(x))^2\mid x\bigr]
=\sigma^2(x)
+\bigl(f^*(x)-\mathbb E_D[\widehat f_D(x)]\bigr)^2
+\operatorname{Var}_D(\widehat f_D(x)).
}
$$

The terms are irreducible noise, squared bias, and variance. Averaging this identity over $X$ gives population prediction risk; applying it to a vector of fixed design points gives the norm form shown in the slides.

More complex models often have lower approximation bias but higher sampling variance. Regularization intentionally increases bias while attempting to reduce variance by a larger amount. OLS is unbiased under the correctly specified fixed-design model, and Gauss-Markov makes it the minimum-variance **linear unbiased** estimator, but a biased estimator may still have smaller total test MSE.

*Sources: L4 slides 30-31 and 34; Bach, Proposition 3.3.*


## 19. Exact fixed-design bias and variance of ridge

Use $\widehat\Sigma=X^\top X/N$ and the observation model $y=Xw_*+\varepsilon$. Ridge can be written as

$$
\widehat w_\lambda
=(\widehat\Sigma+\lambda I)^{-1}
\left(\widehat\Sigma w_*+\frac1N X^\top\varepsilon\right).
$$

Therefore

$$
\mathbb E[\widehat w_\lambda]
=(\widehat\Sigma+\lambda I)^{-1}\widehat\Sigma w_*,
$$

and the parameter bias is

$$
\operatorname{Bias}(\widehat w_\lambda)
=-\lambda(\widehat\Sigma+\lambda I)^{-1}w_*.
$$

Ridge is generally biased whenever $\lambda>0$. Its covariance is

$$
\operatorname{Cov}(\widehat w_\lambda)
=\frac{\sigma^2}{N}
(\widehat\Sigma+\lambda I)^{-1}
\widehat\Sigma
(\widehat\Sigma+\lambda I)^{-1}.
$$

In fixed-design prediction risk, the exact squared-bias and variance terms are

$$
B_\lambda
=\lambda^2w_*^\top
(\widehat\Sigma+\lambda I)^{-1}
\widehat\Sigma
(\widehat\Sigma+\lambda I)^{-1}w_*,
$$

$$
V_\lambda
=\frac{\sigma^2}{N}
\operatorname{tr}\left[
\widehat\Sigma^2(\widehat\Sigma+\lambda I)^{-2}
\right].
$$

If $\widehat\Sigma v_j=\mu_jv_j$, then

$$
B_\lambda=\sum_j
\mu_j\left(\frac{\lambda}{\mu_j+\lambda}\right)^2(v_j^\top w_*)^2,
\qquad
V_\lambda=\frac{\sigma^2}{N}\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

As $\lambda$ grows, variance decreases while bias generally increases. The best predictive $\lambda$ is usually neither zero nor infinity.

*Sources: L4 slides 32-34; Bach, Proposition 3.7.*


## 20. Choosing $\lambda$, treating the intercept, and the MAP view

A theoretical upper bound from the fixed-design analysis is

$$
\mathbb E[R_X(\widehat w_\lambda)]-R_X^*
\le
\frac\lambda2\|w_*\|_2^2
+\frac{\sigma^2\operatorname{tr}(\widehat\Sigma)}{2\lambda N}.
$$

Minimizing this bound suggests

$$
\lambda_*
=\frac{\sigma\sqrt{\operatorname{tr}(\widehat\Sigma)}}
{\|w_*\|_2\sqrt N},
$$

but $w_*$ and $\sigma$ are unknown and the bound need not be tight. In practice, choose $\lambda$ with a validation set or cross-validation, evaluating the metric that matters at test time. Never tune $\lambda$ on the final test set.

Ridge is not invariant to feature units: multiplying one feature by a constant changes how strongly its coefficient is penalized. Standardize nonconstant features using training-set statistics before tuning $\lambda$, then apply the same transformation to validation and test data.

The intercept is usually not penalized. Either center $X$ and $y$, fit ridge without an intercept, and transform back, or use a penalty matrix such as

$$
\lambda w^\top
\operatorname{diag}(0,1,\ldots,1)w.
$$

Ridge also has two interpretations:

- constrained optimization: minimize training loss subject to $\|w\|_2\le t$;
- Bayesian MAP: Gaussian likelihood plus a zero-mean isotropic Gaussian prior on $w$.

The exact mapping between prior variance and $\lambda$ depends on the loss scaling convention.

*Sources: L4 slides 35-36; Bach, Proposition 3.8 and Section 14.1.*


## 21. General regularization, LASSO, and soft-thresholding

Regularized ERM has the general form

$$
\min_w\frac1N\sum_{i=1}^N
(y_i-\phi(x_i)^\top w)^2+\lambda\Omega(w).
$$

The regularizer expresses which solutions are preferred:

- $\Omega(w)=\|w\|_2^2$: ridge; smooth shrinkage and numerical stability;
- $\Omega(w)=\|w\|_1$: LASSO; promotes sparse coefficients;
- group norms: select or shrink predefined groups;
- nuclear norm: promotes low-rank matrices;
- an atomic norm: promotes combinations of a problem-specific set of simple atoms and unifies many structured penalties;
- problem-specific smoothness, graph, or structural penalties.

Choose the penalty to match prior structure: $L_2$ for diffuse small effects, $L_1$ for coordinate sparsity, group norms for block sparsity, the nuclear norm for low rank, and an atomic norm when the natural building blocks are problem-specific atoms.

Ridge has a closed form; LASSO is nonsmooth and generally needs an iterative algorithm. Its basic mechanism is visible in the scalar problem

$$
\min_w\left\{\frac12w^2-xw+\lambda|w|\right\}.
$$

The optimality condition $0\in w-x+\lambda\partial|w|$ gives the soft-threshold solution

$$
w^*=\operatorname{sign}(x)(|x|-\lambda)_+,
\qquad
(a)_+=\max(a,0).
$$

Values with $|x|\le\lambda$ become exactly zero, explaining sparsity. Ridge instead multiplies every orthogonal-coordinate estimate by a continuous factor and normally does not create exact zeros.

Regularization may also be implicit: early stopping, an optimization algorithm's solution preference, architecture, data augmentation, or feature construction can restrict effective model capacity even without an explicit penalty.

*Sources: L4 slides 36-37; solution manual, Chapter 25, Solution 2 (p. 65).*


## 22. A practical end-to-end workflow and common pitfalls

A reliable linear or ridge regression workflow is:

1. Define the prediction target and the test-time distribution.
2. Split data before fitting preprocessing or choosing hyperparameters.
3. Inspect scatter plots, missingness, outliers, leakage, and subgroup structure.
4. Construct features using training data only; include nonlinear features if justified.
5. Center/standardize features when using ridge or LASSO; normally leave the intercept unpenalized.
6. Fit OLS with QR/SVD or a least-squares routine, not an explicit inverse.
7. Tune regularization by validation or cross-validation.
8. Diagnose residual patterns and influential observations.
9. Report both predictive performance and the evaluation design; do not present association as causation.

Common conceptual errors include:

- saying linear regression must be linear in the raw input rather than in parameters;
- assuming small training error implies small generalization error;
- using $\sigma^2d/N$ as an exact random-design test error;
- claiming OLS needs Gaussian noise for unbiasedness;
- comparing numerical $\lambda$ values across differently scaled objectives or features;
- penalizing an intercept accidentally;
- treating a nonunique interpolating solution as though it were uniquely defined;
- interpreting a coefficient causally without a causal design.

*Sources: synthesis of all three sources.*


## 23. Advanced extension: overparameterization, implicit bias, and double descent

When $d>N$ and $X$ has full row rank, there are infinitely many interpolating solutions satisfying $Xw=y$. The minimum-$\ell_2$-norm interpolator is

$$
\widehat w_{\min}=X^+y=X^\top(XX^\top)^{-1}y.
$$

Gradient descent on unregularized squared loss, initialized at $w_0=0$, stays in $\operatorname{row}(X)$ and converges to this minimum-norm solution when it converges. This is an **implicit bias** of the algorithm and initialization; a different ERM selection rule can choose a different interpolator.

For the following exact Gaussian/Wishart formulas, specialize the feature map to $\phi(x)=x$ with $x\sim\mathcal N(0,I_d)$: the features are zero-mean and there is no constant intercept column. If an intercept is fitted, first center the data and adjust the dimensions and degrees of freedom. Under this isotropic Gaussian random design, the expected excess risk of minimum-norm regression has exact expressions away from the interpolation threshold:

$$
d\le N-2:
\qquad
\mathbb E[\text{excess risk}]
=\sigma^2\frac{d}{N-d-1},
$$

$$
d\ge N+2:
\qquad
\mathbb E[\text{excess risk}]
=\sigma^2\frac{N}{d-N-1}
+\|w_*\|_2^2\frac{d-N}{d}.
$$

At the three boundary dimensions, the relevant inverse-Wishart first moment does not exist:

$$
\mathbb E[\text{excess risk}]=\infty,\qquad d\in\{N-1,N,N+1\}.
$$

Thus risk has an infinite interpolation spike in this idealized expectation and can then fall again, producing **double descent**. This does not mean that adding parameters is always beneficial. The phenomenon depends on the data distribution, signal, noise, feature sequence, and implicit or explicit regularization. Properly tuned ridge often smooths the interpolation spike.

*Source: Bach, Sections 12.1-12.2.*


## 24. Advanced extension: random-design prediction risk

In random design, let $\phi(X)$ be a new population feature vector and define

$$
\Sigma=\mathbb E[\phi(X)\phi(X)^\top],
\qquad
\widehat\Sigma=\frac1N X^\top X.
$$

Under a well-specified homoscedastic model, population excess risk is

$$
R(w)-R(w_*)=\|w-w_*\|_\Sigma^2.
$$

Conditional on a full-rank training design, OLS has covariance $(\sigma^2/N)\widehat\Sigma^{-1}$, so

$$
\mathbb E[R(\widehat w)-R(w_*)\mid X]
=\frac{\sigma^2}{N}
\operatorname{tr}(\Sigma\widehat\Sigma^{-1}).
$$

Unlike fixed design, $\Sigma$ and $\widehat\Sigma$ do not cancel. For the exact formula below, specialize to zero-mean Gaussian features $\phi(x)=x\sim\mathcal N(0,\Sigma)$ with no constant intercept column. If an intercept is fitted, center the data first and adjust the dimensions and degrees of freedom. When $N>d+1$, averaging over the Gaussian design gives

$$
\mathbb E[R(\widehat w)-R(w_*)]
=\sigma^2\frac{d}{N-d-1}.
$$

Only when $d/N$ is small is this approximately $\sigma^2d/N$. It diverges as $d$ approaches $N$ from below. This is why the lecture's fixed-design trace calculation proves consistency only in its stated regime; it is not, by itself, a complete unseen-input generalization theorem.

*Source: Bach, Section 3.8.*


## 25. Worked exercises from the slides

**A. Determinant puzzle.** Let $A(t)=\det(C+tD)$ and $M(t)=C+tD$. Jacobi's formula gives, when $M(t)$ is invertible,

$$
A'(t)=\det(C+tD)\operatorname{tr}\bigl((C+tD)^{-1}D\bigr).
$$

A formula valid also at singular points is

$$
A'(t)=\operatorname{tr}\bigl(\operatorname{adj}(C+tD)D\bigr).
$$

**B. Nonlinear features.** With design matrix $\Phi_{i,:}=\phi(x_i)^\top$,

$$
\min_w\|\Phi w-y\|_2^2
\quad\Longrightarrow\quad
\Phi^\top\Phi\widehat w=\Phi^\top y.
$$

If $\Phi$ has full column rank,

$$
\widehat w=(\Phi^\top\Phi)^{-1}\Phi^\top y.
$$

**C. Singular design.** If $X^\top X$ is singular, the minimum-Euclidean-norm OLS solution is $\widehat w_{\min}=X^+y$. Every OLS solution has the form $X^+y+v$ with $v\in\operatorname{null}(X)$, and the fitted-value projector is $XX^+$. With $r=\operatorname{rank}(X)$,

$$
\mathbb E\left[\frac1N\|X\widehat w-Xw_*\|_2^2\right]
=\frac{\sigma^2r}{N}.
$$

**D. Ridge bias and variance.** From Section 19,

$$
\mathbb E[\widehat w_\lambda]-w_*
=-\lambda(\widehat\Sigma+\lambda I)^{-1}w_*,
$$

$$
\operatorname{Cov}(\widehat w_\lambda)
=\frac{\sigma^2}{N}
(\widehat\Sigma+\lambda I)^{-1}\widehat\Sigma
(\widehat\Sigma+\lambda I)^{-1}.
$$

These complete the slide hints rather than merely stating the answers.

*Sources: L4 administration puzzle and slides 16, 25, 32-33.*


## 26. Final checklist and formula sheet

After studying this notebook, you should be able to derive or explain each item below.

1. Squared-loss ERM and alternative losses for median or quantile regression.
2. Simple-regression slope $\widehat w=r s_y/s_x$ and intercept $\bar y-\widehat w\bar x$.
3. Regression to the mean and $\operatorname{RMSE}=s_y\sqrt{1-r^2}$.
4. Why the conditional mean minimizes population squared loss.
5. Why linear in parameters does not mean linear in the raw input.
6. OLS normal equations and the projection interpretation.
7. The difference between fixed and random design.
8. OLS unbiasedness, covariance, and fixed-design excess risk:

$$
\mathbb E[\widehat w]=w_*,
\qquad
\operatorname{Cov}(\widehat w)=\sigma^2(X^\top X)^{-1},
\qquad
\mathbb E[\text{excess risk}]=\frac{\sigma^2d}{N}.
$$

9. The training optimism gap $2\sigma^2d/N$ and the estimator $\mathrm{RSS}/(N-d)$.
10. Pseudoinverse OLS when the parameter solution is nonunique.
11. Ridge under this notebook's scaling:

$$
\widehat w_\lambda=(X^\top X+N\lambda I)^{-1}X^\top y.
$$

12. Spectral shrinkage, the ridge dual form, and effective degrees of freedom.
13. The exact ridge squared-bias and variance terms.
14. Why feature scaling, intercept treatment, and validation matter.
15. How ridge, LASSO, and implicit regularization differ.
16. Why fixed-design formulas do not automatically prove unseen-input generalization.

**Recommended review order:** re-derive Sections 3, 6, 8, 13, 16, and 19 on paper; then solve the four exercises in Section 25 without looking at the displayed answers.
